In [3]:
from bs4 import BeautifulSoup
import requests


# Standard headers to fetch a website
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]


def fetch_website_links(url):
    """
    Return the links on the webiste at the given url
    I realize this is inefficient as we're parsing twice! This is to keep the code in the lab simple.
    Feel free to use a class and optimize it!
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    links = [link.get("href") for link in soup.find_all("a")]
    return [link for link in links if link]


In [4]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

In [2]:
!ollama pull llama3.2:1b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling 74701a8c35f6:   0% ▕                  ▏ 560 KB/1.3 GB                  pulling manifest 
pulling 74701a8c35f6:   0% ▕                  ▏ 1.9 MB/1.3 GB                  pulling manifest 
pulling 74701a8c35f6:   0% ▕                  ▏ 2.6 MB/1.3 GB                  pulling manifest 
pulling 74701a8c35f6:   0% ▕                  ▏ 5.6 MB/1.3 GB                  pulling manifest 
pulling 74701a8c35f6:   1% ▕                  ▏ 8.2 MB/1.3 GB                  pulling manifest 
pulling 74701a8c35f6:   1% ▕                  ▏ 9.5 MB/1.3 GB                  pulling manifest 
pulling 74701a8c35f6:   1% ▕                  ▏  11 MB/1.3

In [5]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL , api_key= "ollama")

In [6]:
gov = "https://www.nationalgeographic.com/history/article/king-tutankhamun"

In [8]:


system_prompt  = """
ou are a Viral Content Specialist and Storyteller. Your goal is to ignore the mundane updates and zero in on the extraordinary, mind-blowing, or emotionally compelling elements within the provided website.

Instructions for analysis:

The 'Wow' Factor (Headline): Start with one shocking or deeply impressive headline that captures the most incredible thing about this content.

Fascinating Insights (Bullet Points): Identify the 'hidden gems'—the facts, features, or ideas that are most likely to spark wonder or debate. Use high-energy, descriptive adjectives.

The 'Mind-Bender' Questions: Frame the content through 3 intrigue-based questions that challenge the reader's perspective or highlight something 'too good to be true.'

Strict Rule: Skip the 'what's new' and focus on the 'what's amazing.' If the information isn't surprising or impressive, find the most unique angle to present it
"""

user_prompt = '''
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

'''

In [9]:
def message_for (website) : 
  return [ 
    {"role" : "system" , "content" : system_prompt},
    {"role" : "user" , "content" : user_prompt + website}
  ]

In [10]:
def summarize ( url ) :
  website = fetch_website_contents (url) 
  response = ollama.chat.completions.create (

    model = "llama3.2:1b" , 
    messages = message_for(website)
  
  )
  return response.choices[0].message.content

In [12]:
def display_summary ( url ) : 
  summary = summarize(url)
  display (Markdown(summary))

In [13]:
display_summary(gov)

I've found some extraordinary elements in this website that will surely capture your attention:


Summary:

King Tut, also known as Tutankhamun, is one of the most famous rulers in ancient Egyptian history. This mysterious 'teen pharaoh' ruled Egypt from 1332 to 1323 BC, leaving behind a legacy that has sparked imagination and fascination for centuries.


News/Announcements:
There are no breaking news or announcements mentioned on this website, apart from being about King Tut's discovery of over 200 years ago.


King Tut Facts & Info | National Geographic:

This section offers an engaging blend of history, culture, and facts about King Tut. Here are some fascinating insights to share:


• **Unique Name**: Some sources describe his birth name as "Tutankhaten," but most historians agree that he was originally known as Tut or "the living image" for short.


• **Imperial Succession**: The rise of 'King' Tut follows the tumultuous period in ancient Egypt, where King Amenhotep IV turned away from traditional Egyptian practices to worship a new sun god. This marked a significant shift towards a monotheistic faith.


• **Golden Life Expectancy**: As reported by the National Geographic, historians believe that young pharaohs like Tut were not always known for their longevity. In fact, he might have lived into his mid-to-late 20s.

The following two points need careful interpretation or slight tweaks to make them intriguing:


• **Mysterious Tomb Discovery**: The discovery of King Tut's tomb is one of the most remarkable stories in history. However, some experts argue that this event has been sensationalized and may not entirely align with historical facts.


• **Golden Mask Surprise**: Perhaps the real 'wow' factor about King Tut stems from the stunning golden mask, which remains an unforgettable sight today. It offers a glimpse into the intricate craftsmanship of ancient Egyptian mortuary practices:


Mind-Bender Questions:

These three questions will challenge your perspective and spark curiosity in your readers:


1. **What drove Egypt's shifting gods?** How did King Akhenaten's radical changes to traditional worship affect the kingdom and its inhabitants?

2. **What secrets lie behind Tutankhamun's mask?** How has this extraordinary artifact enabled historians to re-imagine ancient Egyptian funerary practices and gain insights into mortality rituals?

3. **Would 'King' Tut have remained true to his royal bloodline?: What happened when he ascended the throne at 17 years of age, and did his decision reflect the old ways or a different ambition?